# Script 06 — Versão com lacunas para os alunos

Este notebook foi estruturado como atividade orientada.  
A ideia é que os alunos usem o script Python original como referência e preencham os trechos faltantes.

## Como usar
- execute as células em ordem;
- complete os blocos marcados com `# TODO`;
- adapte o código do arquivo `.py` para o formato notebook;
- compare seus resultados com os arquivos gerados ao final.

Os pontos em aberto estão concentrados nas `defs` e em trechos centrais do pipeline.

In [1]:
pip install pandas numpy matplotlib seaborn joblib datetime scikit-learn xgboost optuna

Note: you may need to restart the kernel to use updated packages.


In [21]:
# Bibliotecas principais
import os
import warnings
from datetime import datetime

import joblib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc
)

import xgboost as xgb

import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")

optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

In [25]:
# Caminhos e parâmetros do experimento
DATASET_FILE = "../csv/ecg_silver_knn_imputado_classified.csv"
OUTPUT_DIR = "../resultados_e_metricas/"
PLOTS_DIR = "../resultados_e_metricas/plots_comparativos/"

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

# Garante que as pastas de saída existam
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Lista de atributos tabulares usados pelos modelos
FEATURE_COLS = [
    "HR", "Pd", "PR", "QRS_Dur", "QT", "QTC",
    "P_axis", "QRS_axis", "T_axis",
    "RV5", "SV1", "RV5_SV1_sum", "RV6", "SV2"
]

# Coluna-alvo
TARGET_COL = "classificacao"

# Faixas fisiológicas ajustadas para permitir valores patológicos biologicamente plausíveis
PHYSIOLOGICAL_RANGES = {
    "HR": (25, 300),
    "Pd": (40, 200),
    "PR": (50, 400),
    "QRS_Dur": (40, 250),
    "QT": (200, 700),
    "QTC": (250, 700),
    "P_axis": (-90, 120),
    "QRS_axis": (-180, 180),
    "T_axis": (-180, 180),
    "RV5": (0, 15),
    "SV1": (0, 15),
    "RV5_SV1_sum": (0, 25),
    "RV6": (0, 15),
    "SV2": (0, 15)
}

RANDOM_STATE = 42
METRIC       = "roc_auc"   # troque por "f1" se preferir
N_TRIALS     = 50
CV_FOLDS     = 5

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


print("Configuração carregada com sucesso.")

Configuração carregada com sucesso.


## Funções para completar

Os blocos abaixo foram deixados propositalmente incompletos.  
O aluno deve preencher cada função consultando o script Python original.

In [22]:
def log_message(message):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] - {message}")


def clip_outliers(df, ranges_dict, output_file=None):
    log_message("Analisando e tratando outliers com ranges ajustados (v1.2)...")

    df_clipped = df.copy()
    outliers_stats = {}
    report_lines = []

    report_lines.append("=" * 80)
    report_lines.append("RELATORIO DE TRATAMENTO DE OUTLIERS (v1.2)")
    report_lines.append("=" * 80)
    report_lines.append("\nVersao: Script v1.2 - Ranges Ajustados para Patologias Reais")
    report_lines.append(f"Dataset: {len(df)} registros")
    report_lines.append(f"Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    total_outliers = 0

    for col, (min_val, max_val) in ranges_dict.items():
        below_min = (df[col] < min_val).sum()
        above_max = (df[col] > max_val).sum()
        n_outliers = below_min + above_max

        if n_outliers > 0:
            original_min = df[col].min()
            original_max = df[col].max()

            df_clipped[col] = df_clipped[col].clip(lower=min_val, upper=max_val)

            outliers_stats[col] = {
                "n_outliers": n_outliers,
                "percent": n_outliers / len(df) * 100,
                "below_min": below_min,
                "above_max": above_max,
                "original_min": original_min,
                "original_max": original_max,
                "range_min": min_val,
                "range_max": max_val
            }

            total_outliers += n_outliers

            report_lines.append(f"\n{'-' * 80}")
            report_lines.append(f"Parametro: {col}")
            report_lines.append(f"  Range ajustado: [{min_val}, {max_val}]")
            report_lines.append(
                f"  Outliers encontrados: {n_outliers} ({n_outliers/len(df)*100:.2f}%)"
            )
            if below_min > 0:
                report_lines.append(
                    f"    - Abaixo do minimo: {below_min} "
                    f"(valor min original: {original_min:.2f})"
                )
            if above_max > 0:
                report_lines.append(
                    f"    - Acima do maximo: {above_max} "
                    f"(valor max original: {original_max:.2f})"
                )
            report_lines.append(
                f"  Acao: Valores clipados para range [{min_val}, {max_val}]"
            )

    report_lines.append(f"\n{'=' * 80}")
    report_lines.append("RESUMO")
    report_lines.append(f"{'=' * 80}")
    report_lines.append(f"Total de parametros com outliers: {len(outliers_stats)}/14")
    report_lines.append(f"Total de valores clipados: {total_outliers}")
    report_lines.append(
        f"Percentual de valores afetados: {total_outliers/(len(df)*14)*100:.2f}%"
    )
    report_lines.append(f"\nTodos os registros foram mantidos (N={len(df)})")
    report_lines.append("Metodo aplicado: CLIPPING (valores limitados aos ranges ajustados)")
    report_lines.append("\nRanges v1.2 incluem valores patologicos mas biologicamente possiveis")

    if output_file:
        with open(output_file, "w", encoding="utf-8") as f:
            f.write("\n".join(report_lines))
        log_message(f"Relatorio de outliers salvo: [{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] - {output_file}")

    if outliers_stats:
        log_message(
            f"Outliers tratados: {len(outliers_stats)} parametros, "
            f"{total_outliers} valores clipados"
        )
        for col, stats in outliers_stats.items():
            log_message(
                f"  {col}: {stats['n_outliers']} outliers ({stats['percent']:.2f}%)"
            )
    else:
        log_message("Nenhum outlier encontrado! Dataset dentro dos ranges ajustados.")

    return df_clipped, outliers_stats


def validate_dataset(df):
    log_message("Validando estrutura do dataset...")

    assert len(df.columns) == 16, f"Esperado 16 colunas, encontrado {len(df.columns)}"
    assert TARGET_COL in df.columns, "Coluna 'classificacao' ausente"
    assert df[TARGET_COL].dtype == "int64", (
        f"Classificacao deve ser int64, encontrado {df[TARGET_COL].dtype}"
    )

    unique_values = set(df[TARGET_COL].unique())
    assert unique_values == {0, 1}, (
        f"Classificacao deve conter apenas {{0, 1}}, encontrado {unique_values}"
    )

    assert df[TARGET_COL].isnull().sum() == 0, "Encontrados NaN em classificacao"

    nan_counts = df[FEATURE_COLS].isnull().sum()
    if nan_counts.sum() > 0:
        log_message("AVISO: NaN encontrados nas features:")
        for col, count in nan_counts[nan_counts > 0].items():
            log_message(f"  {col}: {count} NaN ({count/len(df)*100:.2f}%)")
        raise ValueError("Dataset contem NaN nas features. Verificar etapa de imputacao.")

    log_message("Estrutura validada com sucesso!")
    return True
 

def load_and_prepare_data():
    log_message(f"Carregando o dataset {DATASET_FILE}")
    
    df = pd.read_csv(DATASET_FILE)
    validate_dataset(df)

    df_clean, outliers_stats = clip_outliers(df, PHYSIOLOGICAL_RANGES, OUTPUT_DIR + "outliers_stats.txt")

    X = df_clean[FEATURE_COLS].copy()
    y = df_clean[TARGET_COL].copy()

    # Conjunto de testes
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, 
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    # Conjunto de treino e validação
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size = VAL_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_temp
    )

    log_message(f"Train set: {len(X_train)} registros ({len(X_train)/len(X)*100:.1f}%)")
    log_message(f"Val set:   {len(X_val)} registros ({len(X_val)/len(X)*100:.1f}%)")
    log_message(f"Test set:  {len(X_test)} registros ({len(X_test)/len(X)*100:.1f}%)")

    log_message("Aplicação do scaler")

    scaler = StandardScaler()
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    log_message(f"Media apos escala (train): {X_train_scaled.mean():.4f} (esperado ~0)")
    log_message(f"Desvio padrao apos escala (train): {X_train_scaled.std():.4f} (esperado ~1)")

    return (
        X_train_scaled, X_val_scaled, X_test_scaled,
        y_train, y_val, y_test,
        FEATURE_COLS, scaler, outliers_stats
    )
 


In [23]:
X_train, X_val, X_test, y_train, y_val, y_test, feature_names, scaler, outliers_stats = load_and_prepare_data()

[2026-04-17 08:10:37] - Carregando o dataset ../data/ecg_silver_knn_imputado_classified.csv
[2026-04-17 08:10:37] - Validando estrutura do dataset...
[2026-04-17 08:10:37] - Estrutura validada com sucesso!
[2026-04-17 08:10:37] - Analisando e tratando outliers com ranges ajustados (v1.2)...
[2026-04-17 08:10:37] - Relatorio de outliers salvo: [2026-04-17 08:10:37] - ../resultados_e_metricas/outliers_stats.txt
[2026-04-17 08:10:37] - Outliers tratados: 2 parametros, 85 valores clipados
[2026-04-17 08:10:37] -   HR: 43 outliers (1.24%)
[2026-04-17 08:10:37] -   QTC: 42 outliers (1.21%)
[2026-04-17 08:10:37] - Train set: 2514 registros (72.2%)
[2026-04-17 08:10:37] - Val set:   444 registros (12.8%)
[2026-04-17 08:10:37] - Test set:  523 registros (15.0%)
[2026-04-17 08:10:37] - Aplicação do scaler
[2026-04-17 08:10:37] - Media apos escala (train): 0.0000 (esperado ~0)
[2026-04-17 08:10:37] - Desvio padrao apos escala (train): 0.9258 (esperado ~1)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_pipeline_objective(model_fn):
    """Envolve qualquer modelo num Pipeline com StandardScaler."""
    def objective(trial, X, y):
        model = model_fn(trial)
        pipe  = Pipeline([("scaler", StandardScaler()), ("model", model)])
        return cross_val_score(pipe, X, y, cv=cv, scoring=METRIC).mean()
    return objective


# objetivos agora retornam só o modelo, o Pipeline é aplicado automaticamente
def make_lr(trial):
    return LogisticRegression(
        C      = trial.suggest_float("C", 1e-3, 1e2, log=True),
        penalty= trial.suggest_categorical("penalty", ["l1", "l2"]),
        solver ="liblinear", max_iter=1000, random_state=RANDOM_STATE,
    )

def make_svm(trial):
    return SVC(
        C      = trial.suggest_float("C", 1e-2, 1e3, log=True),
        kernel = trial.suggest_categorical("kernel", ["rbf", "linear"]),
        gamma  = trial.suggest_categorical("gamma", ["scale", "auto"]),
        probability=True, random_state=RANDOM_STATE,
    )

def make_mlp(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers   = tuple(trial.suggest_int(f"units_l{i}", 32, 128) for i in range(n_layers))
    return MLPClassifier(
        hidden_layer_sizes = layers,
        activation         = trial.suggest_categorical("activation", ["relu", "tanh"]),
        learning_rate_init = trial.suggest_float("lr", 1e-4, 1e-2, log=True),
        alpha              = trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
        solver="sgd", learning_rate="adaptive", nesterovs_momentum=True,
        early_stopping=True, max_iter=1000, random_state=RANDOM_STATE,
    )

# XGBoost e GradientBoosting não precisam de scaler, mas o Pipeline não os prejudica
def make_xgb(trial):
    return xgb.XGBClassifier(
        n_estimators     = trial.suggest_int("n_estimators", 100, 500),
        max_depth        = trial.suggest_int("max_depth", 3, 6),
        learning_rate    = trial.suggest_float("lr", 0.05, 0.3),
        subsample        = trial.suggest_float("subsample", 0.6, 0.9),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 0.9),
        min_child_weight = trial.suggest_int("min_child_weight", 5, 20),
        reg_alpha        = trial.suggest_float("reg_alpha", 0.1, 10.0),
        reg_lambda       = trial.suggest_float("reg_lambda", 0.1, 10.0),
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
    )

def make_gb(trial):
    return GradientBoostingClassifier(
        n_estimators     = trial.suggest_int("n_estimators", 100, 400),
        max_depth        = trial.suggest_int("max_depth", 3, 6),
        learning_rate    = trial.suggest_float("lr", 0.05, 0.3),
        subsample        = trial.suggest_float("subsample", 0.6, 0.9),
        min_samples_split= trial.suggest_int("min_samples_split", 10, 30),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 5, 20),
        random_state     = RANDOM_STATE,
    )

def make_rf(trial):
    return RandomForestClassifier(
        n_estimators     = trial.suggest_int("n_estimators", 100, 400),
        max_depth        = trial.suggest_int("max_depth", 5, 20),
        min_samples_split= trial.suggest_int("min_samples_split", 10, 30),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 5, 20),
        max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        random_state     = RANDOM_STATE, n_jobs=-1,
    )


OBJECTIVES = {
    "Logistic Regression": make_pipeline_objective(make_lr),
    "Random Forest":       make_pipeline_objective(make_rf),
    "XGBoost":             make_pipeline_objective(make_xgb),
    "Gradient Boosting":   make_pipeline_objective(make_gb),
    "SVM":                 make_pipeline_objective(make_svm),
    "MLP":                 make_pipeline_objective(make_mlp),
}

def tune_all_models(X_train, y_train, n_trials=N_TRIALS):
    log_message("OTIMIZAÇÃO DE HIPERPARÂMETROS (Optuna)")
    best_params = {}

    for name, objective in OBJECTIVES.items():
        log_message(f"  Otimizando {name}...")
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=n_trials, show_progress_bar=False)

        best_params[name] = study.best_params
        log_message(f"  ✔ {name}: {METRIC} = {study.best_value:.4f} | params: {study.best_params}")

    return best_params


# ─────────────────────────────────────────────
# BUILD — reconstrói modelos com params ótimos
# ─────────────────────────────────────────────

def build_optimized_models(best_params, y_train):
    p = best_params  # atalho

    layers = tuple(
        p["MLP"][f"units_l{i}"]
        for i in range(p["MLP"]["n_layers"])
    )

    return {
        "Logistic Regression": LogisticRegression(
            **{k: v for k, v in p["Logistic Regression"].items()},
            class_weight="balanced", solver="liblinear",
            max_iter=1000, random_state=RANDOM_STATE,
        ),
        "Random Forest": RandomForestClassifier(
            **{k: v for k, v in p["Random Forest"].items()},
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "XGBoost": xgb.XGBClassifier(
            **{k: v for k, v in p["XGBoost"].items()},
            scale_pos_weight=sum(y_train == 0) / sum(y_train == 1),
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            **{k: v for k, v in p["Gradient Boosting"].items()},
            random_state=RANDOM_STATE,
        ),
        "SVM": SVC(
            **{k: v for k, v in p["SVM"].items()},
            class_weight="balanced", probability=True, random_state=RANDOM_STATE,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=layers,
            activation=p["MLP"]["activation"],
            learning_rate_init=p["MLP"]["learning_rate_init"],
            alpha=p["MLP"]["alpha"],
            solver="sgd", learning_rate="adaptive", nesterovs_momentum=True,
            early_stopping=True, max_iter=1000, random_state=RANDOM_STATE,
        ),
    }


# ─────────────────────────────────────────────
# EVALUATE  (sem alterações na lógica original)
# ─────────────────────────────────────────────

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall":    recall_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred),
        "auc_roc":   roc_auc_score(y_test, y_prob),
    }
    return metrics, y_pred, y_prob


def save_all_results(results, trained_models, best_params, timestamp=True):
    """
    Salva modelos, métricas e hiperparâmetros em results/.

    Estrutura gerada:
        results_YYYYMMDD_HHMMSS/
        ├── models/          → .pkl de cada modelo
        ├── metrics.csv      → tabela comparativa
        └── hyperparams.json → melhores params do Optuna
    """
    suffix = f"_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}" if timestamp else ""
    base   = OUTPUT_DIR + suffix
    models_dir = os.path.join(base, "models")

    os.makedirs(models_dir, exist_ok=True)

    # ── 1. MODELOS (.pkl) ──────────────────────────────────────────
    for name, data in trained_models.items():
        filename = name.lower().replace(" ", "_") + ".pkl"
        filepath = os.path.join(models_dir, filename)
        joblib.dump(data["model"], filepath)
        log_message(f"  Modelo salvo: {filepath}")

    # ── 2. MÉTRICAS (CSV) ──────────────────────────────────────────
    df_metrics = (
        pd.DataFrame(results)
        .T
        .rename_axis("model")
        .reset_index()
        .sort_values("auc_roc", ascending=False)
    )
    metrics_path = os.path.join(base, "metrics.csv")
    df_metrics.to_csv(metrics_path, index=False, float_format="%.4f")
    log_message(f"  Métricas salvas: {metrics_path}")

    # ── 3. HIPERPARÂMETROS (JSON) ──────────────────────────────────
    params_path = os.path.join(base, "hyperparams.json")
    with open(params_path, "w") as f:
        json.dump(best_params, f, indent=2)
    log_message(f"  Hiperparâmetros salvos: {params_path}")

    log_message(f"✔ Resultados salvos em: {base}/")
    return base

# ─────────────────────────────────────────────
# TRAIN & EVALUATE — entry point principal
# ─────────────────────────────────────────────

def train_and_evaluate_all(X_train, y_train, X_val, y_val, X_test, y_test):
    # 1) Otimiza hiperparâmetros com cross-validation no treino
    best_params = tune_all_models(X_train, y_train)

    # 2) Reconstrói modelos com os melhores parâmetros
    models = build_optimized_models(best_params, y_train)

    log_message("TREINAMENTO DOS MODELOS")
    results        = {}
    trained_models = {}

    for name, model in models.items():
        log_message(f"Treinando o modelo {name}")

        # XGBoost usa X_val para early stopping
        if name == "XGBoost":
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        else:
            model.fit(X_train, y_train)

        metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test)

        log_message(f"  Accuracy:  {metrics['accuracy']:.4f}")
        log_message(f"  Precision: {metrics['precision']:.4f}")
        log_message(f"  Recall:    {metrics['recall']:.4f}")
        log_message(f"  F1-Score:  {metrics['f1']:.4f}")
        log_message(f"  AUC-ROC:   {metrics['auc_roc']:.4f}")

        results[name]        = metrics
        trained_models[name] = {"model": model, "y_pred": y_pred, "y_proba": y_proba}

    # 3) Ranking final
    log_message("RANKING FINAL (AUC-ROC)")
    ranking = sorted(results.items(), key=lambda x: x[1]["auc_roc"], reverse=True)
    for i, (name, m) in enumerate(ranking, 1):
        log_message(f"  {i}. {name}: AUC-ROC={m['auc_roc']:.4f}  F1={m['f1']:.4f}")

    save_all_results(results, trained_models, best_params)

    return results, trained_models, best_params

In [ ]:
# TODO: após implementar get_models, evaluate_model e train_and_evaluate_all
results, trained_models, best_params = train_and_evaluate_all(X_train, y_train, X_val, y_val, X_test, y_test)

[2026-04-16 14:38:15] - OTIMIZAÇÃO DE HIPERPARÂMETROS (Optuna)
[2026-04-16 14:38:15] -   Otimizando Logistic Regression...
[2026-04-16 14:38:18] -   ✔ Logistic Regression: roc_auc = 0.7712 | params: {'C': 1.943709682639589, 'penalty': 'l2'}
[2026-04-16 14:38:18] -   Otimizando Random Forest...
[2026-04-16 14:45:39] -   ✔ Random Forest: roc_auc = 0.9260 | params: {'n_estimators': 532, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
[2026-04-16 14:45:39] -   Otimizando XGBoost...
[2026-04-16 14:49:01] -   ✔ XGBoost: roc_auc = 0.9221 | params: {'n_estimators': 691, 'max_depth': 10, 'learning_rate': 0.013754368392643398, 'subsample': 0.7805907593538831, 'colsample_bytree': 0.7118159541746358, 'colsample_bylevel': 0.8083939899971984, 'colsample_bynode': 0.8675601695146626, 'reg_alpha': 0.13921739669930935, 'reg_lambda': 0.009631562250142358}
[2026-04-16 14:49:01] -   Otimizando Gradient Boosting...
[2026-04-16 15:08:06] -   ✔ Gradient Boosting: roc_au

NameError: name 'json' is not defined

In [78]:

save_all_results(results, trained_models, best_params)
# print(best_params)

[2026-04-16 15:37:15] -   Modelo salvo: ../resultados_e_metricas/_20260416_153715/models/logistic_regression.pkl
[2026-04-16 15:37:16] -   Modelo salvo: ../resultados_e_metricas/_20260416_153715/models/random_forest.pkl
[2026-04-16 15:37:16] -   Modelo salvo: ../resultados_e_metricas/_20260416_153715/models/xgboost.pkl
[2026-04-16 15:37:16] -   Modelo salvo: ../resultados_e_metricas/_20260416_153715/models/gradient_boosting.pkl
[2026-04-16 15:37:16] -   Modelo salvo: ../resultados_e_metricas/_20260416_153715/models/svm.pkl
[2026-04-16 15:37:16] -   Modelo salvo: ../resultados_e_metricas/_20260416_153715/models/mlp.pkl
[2026-04-16 15:37:16] -   Métricas salvas: ../resultados_e_metricas/_20260416_153715/metrics.csv
[2026-04-16 15:37:16] -   Hiperparâmetros salvos: ../resultados_e_metricas/_20260416_153715/hyperparams.json
[2026-04-16 15:37:16] - ✔ Resultados salvos em: ../resultados_e_metricas/_20260416_153715/


'../resultados_e_metricas/_20260416_153715'

In [ ]:
df_results = pd.DataFrame(best_params).T

print(df_results)

df_results.to_csv(OUTPUT_DIR + "baseline_results.csv", index=True)
joblib.dump(trained_models["XGBoost"]["model"], OUTPUT_DIR + "best_model_XGB.pkl")
joblib.dump(scaler, OUTPUT_DIR + "scaler.pkl")

log_message(f"Resultados salvos: {OUTPUT_DIR}baseline_results.csv")
log_message(f"Melhor modelo salvo: {OUTPUT_DIR}best_model_XGB.pkl")
log_message(f"Scaler salvo: {OUTPUT_DIR}scaler.pkl")

# df_results.to_csv(OUTPUT_DIR + "best_params.csv")
# df_results = df_results[["accuracy", "precision", "recall", "f1", "auc_roc"]]
# df_results

                            C penalty n_estimators max_depth  \
Logistic Regression   1.94371      l2          NaN       NaN   
Random Forest             NaN     NaN          532        30   
XGBoost                   NaN     NaN        691.0      10.0   
Gradient Boosting         NaN     NaN        504.0       5.0   
SVM                  4.468158     NaN          NaN       NaN   
MLP                       NaN     NaN          NaN       NaN   

                    min_samples_split min_samples_leaf max_features  \
Logistic Regression               NaN              NaN          NaN   
Random Forest                       2                1         log2   
XGBoost                           NaN              NaN          NaN   
Gradient Boosting                14.0              3.0          NaN   
SVM                               NaN              NaN          NaN   
MLP                               NaN              NaN          NaN   

                    learning_rate subsample colsample

In [15]:
import os
import json
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ─────────────────────────────────────────────
# CARREGA hiperparâmetros salvos
# ─────────────────────────────────────────────

def load_best_params(results_dir):
    params_path = os.path.join(results_dir, "hyperparams.json")
    with open(params_path, "r") as f:
        best_params = json.load(f)
    log_message(f"Hiperparâmetros carregados de: {params_path}")
    return best_params


# ─────────────────────────────────────────────
# TREINA com os melhores parâmetros
# ─────────────────────────────────────────────

def train_with_best_params(best_params, X_train, y_train, X_val, y_val):
    log_message("TREINAMENTO COM MELHORES PARÂMETROS")

    models = build_optimized_models(best_params, y_train)  # função já existente
    trained_models = {}

    for name, model in models.items():
        log_message(f"  Treinando {name}...")

        if name == "XGBoost":
            model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        else:
            model.fit(X_train, y_train)

        trained_models[name] = model

    return trained_models


# ─────────────────────────────────────────────
# AVALIA e coleta métricas
# ─────────────────────────────────────────────

def evaluate_all(trained_models, X_test, y_test):
    log_message("AVALIAÇÃO DOS MODELOS")

    results = {}

    for name, model in trained_models.items():
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy":  accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall":    recall_score(y_test, y_pred),
            "f1":        f1_score(y_test, y_pred),
            "auc_roc":   roc_auc_score(y_test, y_prob),
        }
        results[name] = metrics

        log_message(f"  {name}")
        for metric, value in metrics.items():
            log_message(f"    {metric}: {value:.4f}")

    return results


# ─────────────────────────────────────────────
# SALVA modelos + métricas
# ─────────────────────────────────────────────

def save_trained_models(trained_models, results, timestamp=True):
    suffix     = f"_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}" if timestamp else ""
    base       = OUTPUT_DIR + suffix
    models_dir = os.path.join(base, "models")
    os.makedirs(models_dir, exist_ok=True)

    # .pkl de cada modelo
    for name, model in trained_models.items():
        filename = name.lower().replace(" ", "_") + ".pkl"
        filepath = os.path.join(models_dir, filename)
        joblib.dump(model, filepath)
        log_message(f"  Modelo salvo: {filepath}")

    # métricas em CSV
    df_metrics = (
        pd.DataFrame(results)
        .T
        .rename_axis("model")
        .reset_index()
        .sort_values("auc_roc", ascending=False)
    )
    metrics_path = os.path.join(base, "metrics.csv")
    df_metrics.to_csv(metrics_path, index=False, float_format="%.4f")
    log_message(f"  Métricas salvas: {metrics_path}")

    log_message(f"✔ Tudo salvo em: {base}/")
    return base


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

def retrain_and_save(results_dir, X_train, y_train, X_val, y_val, X_test, y_test):
    """
    Fluxo completo:
        1. Carrega hiperparâmetros do JSON salvo anteriormente
        2. Treina os modelos com esses parâmetros
        3. Avalia no conjunto de teste
        4. Salva os .pkl e o metrics.csv

    Parâmetros
    ----------
    results_dir : str
        Pasta gerada pelo save_all_results() anterior, ex: "results_20250416_143022"
    """
    best_params    = load_best_params(results_dir)
    trained_models = train_with_best_params(best_params, X_train, y_train, X_val, y_val)
    results        = evaluate_all(trained_models, X_test, y_test)
    output_dir     = save_trained_models(trained_models, results)

    return trained_models, results, output_dir

In [16]:
trained_models, results, output_dir = retrain_and_save(
    results_dir = OUTPUT_DIR + "_20260416_153715",  # pasta do Optuna
    X_train     = X_train,
    y_train     = y_train,
    X_val       = X_val,
    y_val       = y_val,
    X_test      = X_test,
    y_test      = y_test,
)

[2026-04-16 16:09:43] - Hiperparâmetros carregados de: ../resultados_e_metricas/_20260416_153715/hyperparams.json
[2026-04-16 16:09:43] - TREINAMENTO COM MELHORES PARÂMETROS
[2026-04-16 16:09:43] -   Treinando Logistic Regression...
[2026-04-16 16:09:43] -   Treinando Random Forest...
[2026-04-16 16:09:45] -   Treinando XGBoost...
[2026-04-16 16:09:47] -   Treinando Gradient Boosting...
[2026-04-16 16:09:53] -   Treinando SVM...
[2026-04-16 16:09:53] -   Treinando MLP...
[2026-04-16 16:09:56] - AVALIAÇÃO DOS MODELOS
[2026-04-16 16:09:56] -   Logistic Regression
[2026-04-16 16:09:56] -     accuracy: 0.7017
[2026-04-16 16:09:56] -     precision: 0.7017
[2026-04-16 16:09:56] -     recall: 1.0000
[2026-04-16 16:09:56] -     f1: 0.8247
[2026-04-16 16:09:56] -     auc_roc: 0.5000
[2026-04-16 16:09:57] -   Random Forest
[2026-04-16 16:09:57] -     accuracy: 0.7017
[2026-04-16 16:09:57] -     precision: 0.7017
[2026-04-16 16:09:57] -     recall: 1.0000
[2026-04-16 16:09:57] -     f1: 0.8247
[2

In [18]:
df_results = pd.DataFrame(results).T
df_results = df_results[["accuracy", "precision", "recall", "f1", "auc_roc"]]
df_results

,accuracy,precision,recall,f1,auc_roc
Logistic Regression,0.701721,0.701721,1.0,0.824719,0.500000
Random Forest,0.701721,0.701721,1.0,0.824719,0.682422
XGBoost,0.701721,0.701721,1.0,0.824719,0.684247
Gradient Boosting,0.701721,0.701721,1.0,0.824719,0.700500
SVM,0.701721,0.701721,1.0,0.824719,0.500000
MLP,0.701721,0.701721,1.0,0.824719,0.500000


In [20]:
# 1. Cheque o tamanho dos conjuntos — devem ser os mesmos nas duas etapas
print(X_train.shape, X_val.shape, X_test.shape)

# 2. Cheque a proporção das classes
print("Train:", y_train.mean(), "| Val:", y_val.mean(), "| Test:", y_test.mean())

best_params = load_best_params(OUTPUT_DIR + "_20260416_153715")

# 3. Teste o XGBoost sem early stopping para isolar o bug
model = xgb.XGBClassifier(**best_params["XGBoost"])
model.fit(X_train, y_train)  # sem eval_set
print(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

(2514, 14) (444, 14) (523, 14)
Train: 0.7012728719172633 | Val: 0.7004504504504504 | Test: 0.7017208413001912
[2026-04-16 16:23:56] - Hiperparâmetros carregados de: ../resultados_e_metricas/_20260416_153715/hyperparams.json
0.6917138265912107


In [ ]:

def plot_comparison(results, output_path):
    log_message("Gerando grafico de comparacao de modelos...")
    df_results = pd.DataFrame(results).T
    df_results = df_results[["accuracy", "precision", "recall", "f1", "auc_roc"]]

    fig, ax = plt.subplots(figsize=(12, 6))
    df_results.plot(kind="bar", ax=ax, width=0.8)
    plt.title("Comparacao de Performance - Modelos Baseline", fontsize=14, fontweight="bold")
    plt.xlabel("Modelo", fontsize=12)
    plt.ylabel("Score", fontsize=12)
    plt.xticks(rotation=45, ha="right")
    plt.legend(loc="lower right", fontsize=10)
    plt.ylim([0.7, 1.0])
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    log_message(f"Grafico salvo: {output_path}")


def plot_feature_importance(model, feature_names, model_name, output_path):
    log_message(f"Gerando feature importance para {model_name}...")

    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    elif hasattr(model, "coef_"):
        importances = np.abs(model.coef_[0])
    else:
        log_message(f"  {model_name} nao suporta feature importance. Pulando...")
        return

    indices = np.argsort(importances)[::-1]

    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = np.arange(len(importances))
    ax.barh(y_pos, importances[indices], alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([feature_names[i] for i in indices])
    ax.invert_yaxis()
    ax.set_xlabel("Importancia", fontsize=12)
    ax.set_title(f"Feature Importance - {model_name}", fontsize=14, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    log_message(f"Grafico salvo: {output_path}")

    log_message(f"  Top 5 features ({model_name}):")
    for i in range(min(5, len(importances))):
        idx = indices[i]
        log_message(f"    {i+1}. {feature_names[idx]}: {importances[idx]:.4f}")


def plot_confusion_matrix(y_test, y_pred, model_name, output_path):
    log_message(f"Gerando confusion matrix para {model_name}...")
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["NORMAL", "ANORMAL"],
        yticklabels=["NORMAL", "ANORMAL"],
        ax=ax, cbar_kws={"label": "Count"}
    )
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_title(f"Confusion Matrix - {model_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    log_message(f"Grafico salvo: {output_path}")


def plot_roc_curves(trained_models, y_test, output_path):
    log_message("Gerando curvas ROC comparativas...")

    fig, ax = plt.subplots(figsize=(10, 8))

    for name, model_data in trained_models.items():
        y_proba = model_data["y_proba"]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.500)")
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curves - Comparacao de Modelos", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    log_message(f"Grafico salvo: {output_path}")

In [ ]:

plot_comparison(results, PLOTS_DIR + "baseline_comparison.png")

plot_feature_importance(
    trained_models["Random Forest"]["model"],
    feature_names,
    "Random Forest",
    PLOTS_DIR + "feature_importance_RF.png"
)

plot_feature_importance(
    trained_models["XGBoost"]["model"],
    feature_names,
    "XGBoost",
    PLOTS_DIR + "feature_importance_XGB.png"
)

plot_confusion_matrix(
    y_test,
    trained_models["XGBoost"]["y_pred"],
    "XGBoost",
    PLOTS_DIR + "confusion_matrix_XGB.png"
)

plot_roc_curves(
    trained_models,
    y_test,
    PLOTS_DIR + "roc_curves_comparison.png"
)

## Roteiro de execução

Depois de preencher as funções, execute as células abaixo para testar se o notebook ficou funcional.

In [26]:
# Informações gerais do dataset
print(df.shape)
print(df.dtypes)
print(df[TARGET_COL].value_counts(dropna=False))

(3481, 16)
filename             str
HR               float64
Pd               float64
PR               float64
QRS_Dur          float64
QT               float64
QTC              float64
P_axis           float64
QRS_axis         float64
T_axis           float64
RV5              float64
SV1              float64
RV5_SV1_sum      float64
RV6              float64
SV2              float64
classificacao      int64
dtype: object
classificacao
1    2441
0    1040
Name: count, dtype: int64


In [ ]:
# TODO: salvar resultados e modelos
# Dica:
# df_results.to_csv(...)
# joblib.dump(...)
pass

In [ ]:
# TODO: gerar os gráficos finais
# Dica:
# plot_comparison(...)
# plot_feature_importance(...)
# plot_confusion_matrix(...)
# plot_roc_curves(...)
pass